# 3-SAT oracle generation

The complete original seeded sweep uses checked native qfunc synthesis and a reversible Circuit route. The plot identifies each local route and the historical Qiskit points. The measurement and independent amplitude check use different problem sizes because the source sweep benchmarks generation.

## Separate the predicate from its gate realization

The classical formula describes which four-bit assignments are marked. The
quantum program prepares a superposition and uses the formula to assign a minus
sign to satisfying assignments. FlagQuantum constructs the predicate circuit
and cleans temporary work; the diagonal Circuit below independently states the
truth table, rather than reusing that decomposition.

A phase oracle alone leaves assignment probabilities unchanged. Interference in
a search algorithm is what turns marking into a changed readout distribution.
Try another accepted clause and compare marked assignments and resources; update
the independent formula expectation as part of that experiment.


Generate a bounded signed-clause formula by drawing two valid permutations.

In [ ]:
import math
from pathlib import Path

import numpy as np
import torch
from IPython.display import Image, display

import examples.synthesis.tutorials.technology_demonstrations.oracle_generation.three_sat_oracles.overview.benchmark as benchmark
import examples.synthesis.tutorials.technology_demonstrations.oracle_generation.three_sat_oracles.overview.oracle as oracle
import examples.synthesis.tutorials.technology_demonstrations.oracle_generation.three_sat_oracles.overview.plots as plots
import flagquantum as fq
from examples.synthesis.tutorials.technology_demonstrations.oracle_generation.three_sat_oracles.overview.main import (
    print_report,
)

formula = benchmark.generate(4, np.random.RandomState(benchmark.SEED))
print(formula)

Inspect the generated Python predicate and synthesize its exact phase oracle.

In [ ]:
print(oracle.native_source(formula, uniform=True))
program = oracle.build_program(formula, uniform=True)

Check every coherent amplitude against an independent truth-table Circuit.

In [ ]:
phases = torch.tensor([(-1 if formula.satisfies(word) else 1) for word in range(16)], dtype=torch.complex128)
reference = fq.Circuit(4, dtype=torch.complex128)
for wire in range(4):
    reference.h(wire)
reference.any(*range(4), matrix=torch.diag(phases))
expected = fq.run(reference).statevector().flatten()
for candidate in (program, oracle.clause_circuit(formula, uniform=True)):
    actual = fq.run(candidate).statevector().reshape(16, -1)
    torch.testing.assert_close(actual[:, 0], expected, rtol=0, atol=1e-12)
    assert float(actual[:, 1:].abs().square().sum()) < 1e-24
print('Complete coherent reference:', 16)

Use the local monotonic clock and native resource estimator.

In [ ]:
assert math.isclose(float(expected.abs().square().sum()), 1.0, abs_tol=1e-12)

Retain seed 128 and every size from the original rounding formula.

In [ ]:
sizes = benchmark.sizes()
assert (len(sizes), sizes[0], sizes[-1]) == (23, 11, 68)
print(sizes)

The original Qiskit installation and timing loop remain disabled.

In [ ]:
# The historical optional Qiskit and Tweedledum setup is not run.

Construct all 23 oracles and verify their emitted instructions.

In [ ]:
rows = benchmark.run()
print_report(rows)

Keep the 12 archived Qiskit times with their source labels.

In [ ]:
assert len(benchmark.HISTORICAL_QISKIT) == 12

Plot the measured local generation time and physical width.

In [ ]:
figure = Path('notebook-figures') / 'oracle-generation.png'
plots.save(rows, figure)
display(Image(filename=str(figure)))